In [0]:
from pyspark.sql.functions import col, lit, count

# Ingestão das dependências estruturadas (Camadas Bronze e Prata)
df_bronze = spark.table("bronze_openalex")
df_silver_authors = spark.table("silver_authors_flattened")
df_silver_citations = spark.table("silver_citations_flattened")

# Modelagem de Vértices: extração de Trabalhos e Autores com padronização de schema e deduplicação
df_nodes_works = df_bronze.select(
    col("id").alias("id_no"),
    col("title").alias("nome_no"),
    lit("Trabalho").alias("tipo_no")
).dropDuplicates(["id_no"])

df_nodes_authors = df_silver_authors.select(
    col("author_id").alias("id_no"),
    col("author_name").alias("nome_no"),
    lit("Autor").alias("tipo_no")
).dropDuplicates(["id_no"])

# Consolidação do espaço de vértices
df_gold_vertices = df_nodes_works.unionByName(df_nodes_authors)

# Modelagem de Arestas: cálculo de adjacência direcional e peso (frequência) via agregação
df_edges_citations = df_silver_citations.groupBy(
    col("work_id_origem").alias("id_origem"),
    col("work_id_citado").alias("id_destino")
).agg(count("*").alias("peso_aresta")) \
 .withColumn("tipo_aresta", lit("Citação"))

df_edges_authorships = df_silver_authors.groupBy(
    col("author_id").alias("id_origem"),
    col("work_id").alias("id_destino")
).agg(count("*").alias("peso_aresta")) \
 .withColumn("tipo_aresta", lit("Autoria"))

# Consolidação do espaço de arestas
df_gold_arestas = df_edges_citations.unionByName(df_edges_authorships)

# Persistência física em formato Delta e log de auditoria
df_gold_vertices.write.format("delta").mode("overwrite").saveAsTable("gold_vertices")
df_gold_arestas.write.format("delta").mode("overwrite").saveAsTable("gold_arestas")

print(f"Total de Vértices: {df_gold_vertices.count()}")
print(f"Total de Arestas: {df_gold_arestas.count()}")